⚙️ 1. Instalar dependências

In [ ]:
!pip install roboflow torch torchvision matplotlib opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 76.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


📥 2. Baixar dataset (formato COCO)

⚠️ SSD usa melhor COCO

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


📂 3. Estrutura esperada

Depois disso você terá algo tipo:

```
train/
  _annotations.coco.json
valid/
  _annotations.coco.json
```



🧠 4. Criar Dataset customizado (PyTorch COCO)

In [ ]:
import torch
import torchvision
from torchvision.datasets import CocoDetection
from torchvision import transforms


transform = transforms.Compose([
    transforms.Resize((320)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = CocoDetection(
    root=dataset.location + "/train",
    annFile=dataset.location + "/train/_annotations.coco.json",
    transform=transform
)

val_dataset = CocoDetection(
    root=dataset.location + "/valid",
    annFile=dataset.location + "/valid/_annotations.coco.json",
    transform=transform
)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


🔁 5. DataLoader

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

🧠 6. Criar modelo SSD + MobileNetV3

In [ ]:
import torchvision.models.detection as detection

num_classes = len(train_dataset.coco.getCatIds()) + 1  # +1 background

model = detection.ssdlite320_mobilenet_v3_large(pretrained=True)
model.head.classification_head.num_classes = num_classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1`. You can also use `weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            )
          )
        )
        (2): Invert

⚙️ 7. Otimizador

In [ ]:
import torch.optim as optim

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params, lr=0.0003)

🔁 8. Loop de treino

In [ ]:
num_epochs = 50
cat_ids = train_dataset.coco.getCatIds()
cat_id_to_label = {cat_id: i+1 for i, cat_id in enumerate(cat_ids)}

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]

        targets_formatted = []
        for t in targets:
            boxes = []
            labels = []

            for obj in t:
                x, y, w, h = obj['bbox']
                boxes.append([x, y, x+w, y+h])
                labels.append(cat_id_to_label[obj['category_id']])

            if len(boxes) == 0:
              continue

            targets_formatted.append({
                "boxes": torch.tensor(boxes, dtype=torch.float32).to(device),
                "labels": torch.tensor(labels, dtype=torch.int64).to(device)
            })

        loss_dict = model(images, targets_formatted)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 13.278802498527195
Epoch 2, Loss: 10.027365104011867
Epoch 3, Loss: 8.700561440509299
Epoch 4, Loss: 7.540744283924932
Epoch 5, Loss: 6.160633066426152
Epoch 6, Loss: 5.205476677936057
Epoch 7, Loss: 4.426955274913622
Epoch 8, Loss: 3.6558950569318687
Epoch 9, Loss: 2.8378816065580947
Epoch 10, Loss: 2.5940051700757896
Epoch 11, Loss: 2.355758698090263
Epoch 12, Loss: 2.113399733667788
Epoch 13, Loss: 1.9031167133994724
Epoch 14, Loss: 1.6925548366878345
Epoch 15, Loss: 1.5582003541614697
Epoch 16, Loss: 1.4339971853339153
Epoch 17, Loss: 1.261379361152649
Epoch 18, Loss: 1.2741617793622224
Epoch 19, Loss: 1.12571294411369
Epoch 20, Loss: 1.1218463260194529
Epoch 21, Loss: 1.0376274222912996
Epoch 22, Loss: 1.0026466094929238
Epoch 23, Loss: 0.9540797886641129
Epoch 24, Loss: 0.8228195879770361
Epoch 25, Loss: 0.7737234856771387
Epoch 26, Loss: 0.7313562294711238
Epoch 27, Loss: 0.7193774088569309
Epoch 28, Loss: 0.6902666027131288
Epoch 29, Loss: 0.6365220196869062
Epoc

✅ 9. Teste rápido

In [ ]:
model.eval()

images, _ = next(iter(val_loader))
images = [img.to(device) for img in images]

with torch.no_grad():
    predictions = model(images)

print(predictions[0])

{'boxes': tensor([[320.0000, 144.9729, 320.0000, 320.0000],
        [123.4220, 240.4088, 180.1494, 282.5777],
        [ 19.5862, 320.0000, 168.7570, 320.0000],
        [320.0000, 141.1122, 320.0000, 320.0000],
        [170.3412, 241.8777, 237.3307, 286.3982],
        [  8.6333, 185.0230, 259.6599, 320.0000],
        [158.3790, 223.3059, 222.8031, 272.3916],
        [  0.0000, 245.6335, 212.5627, 320.0000],
        [221.0152, 160.5849, 268.3236, 207.9254],
        [202.6138, 209.0148, 261.2062, 255.8462],
        [242.0128, 199.4688, 294.5711, 251.3420],
        [269.8926, 289.7017, 320.0000, 320.0000],
        [ 46.1513, 291.0683, 194.0395, 320.0000],
        [257.5603, 206.7386, 308.2628, 252.9299],
        [112.1453,  80.3077, 156.0677, 125.7795],
        [197.3634, 217.6204, 244.3868, 256.4663],
        [226.6619, 210.6282, 277.8626, 259.0397],
        [113.2615, 249.3663, 181.0315, 306.0435],
        [175.2260,  55.8446, 310.3646, 170.0616],
        [ 87.0809,  89.3540, 117.1154, 1

💾 10. Salvar modelo

In [ ]:
torch.save(model.state_dict(), "ssd_mobilenetv3.pth")

📱 11. Converter para mobile (opcional)

In [ ]:
model.eval()

scripted_model = torch.jit.script(model)
scripted_model.save("ssd_mobilenetv3_mobile.pt")

✅ Teste o modelo salvo (IMPORTANTE)

Antes de usar no mobile, testa:

In [ ]:
loaded_model = torch.jit.load("ssd_mobilenetv3_mobile.pt")
loaded_model.eval()

example = [torch.rand(3, 320, 320).to(device)]

with torch.no_grad():
    output = loaded_model(example)

print(output)

({}, [{'boxes': tensor([[320.0000, 121.0777, 320.0000, 196.4293],
        [320.0000, 121.0777, 320.0000, 196.4293],
        [320.0000,  56.7620, 320.0000, 124.8692],
        [320.0000,  56.7620, 320.0000, 124.8692],
        [320.0000, 121.0777, 320.0000, 196.4293],
        [320.0000, 195.8152, 320.0000, 267.1158],
        [320.0000, 121.0777, 320.0000, 196.4293],
        [320.0000, 249.2062, 320.0000, 320.0000],
        [320.0000,  56.7620, 320.0000, 124.8692],
        [320.0000, 195.8152, 320.0000, 267.1158],
        [320.0000, 195.8152, 320.0000, 267.1158],
        [320.0000, 204.6613, 320.0000, 269.8791],
        [320.0000, 239.6839, 320.0000, 300.0561]], device='cuda:0'), 'scores': tensor([0.0146, 0.0105, 0.0104, 0.0073, 0.0057, 0.0043, 0.0034, 0.0027, 0.0026,
        0.0024, 0.0016, 0.0011, 0.0010], device='cuda:0'), 'labels': tensor([5, 6, 4, 5, 4, 6, 3, 5, 6, 5, 3, 4, 4], device='cuda:0')}])


SSD always returns a (Losses, Detections) tuple in scripting


# sssss

peso

In [ ]:
import os

size_mb = os.path.getsize("ssd_mobilenetv3.pth") / (1024 * 1024)
print(f"Peso: {size_mb:.2f} MB")

Peso: 13.42 MB


LATÊNCIA + FPS

In [ ]:
import time

model.eval()

image = [torch.rand(3, 320, 320).to(device)]

# warmup
for _ in range(10):
    _ = model(image)

start = time.time()

for _ in range(100):
    _ = model(image)

end = time.time()

latency = (end - start) / 100
fps = 1 / latency

print(f"Latência: {latency*1000:.2f} ms")
print(f"FPS: {fps:.2f}")

Latência: 40.70 ms
FPS: 24.57


🧠 3. GFLOPs (complexidade)

In [ ]:
!pip install thop

In [ ]:
from thop import profile

dummy = torch.rand(1, 3, 320, 320).to(device)

flops, params = profile(model, inputs=([dummy[0]],))
print(f"GFLOPs: {flops / 1e9:.2f}")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU6'>.
GFLOPs: 0.62


🎯 4. mAP@50 (DETecção) — MAIS IMPORTANTE

In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

cat_ids = train_dataset.coco.getCatIds()
cat_id_to_label = {cat_id: i+1 for i, cat_id in enumerate(cat_ids)}

metric = MeanAveragePrecision(iou_thresholds=[0.5])

model.eval()

for images, targets in val_loader:
    images = [img.to(device) for img in images]

    preds = model(images)

    preds_formatted = []
    targets_formatted = []

    for p, t in zip(preds, targets):
        preds_formatted.append({
            "boxes": p["boxes"].cpu(),
            "scores": p["scores"].cpu(),
            "labels": p["labels"].cpu()
        })

        boxes = []
        labels = []

        for obj in t:
            x, y, w, h = obj['bbox']
            boxes.append([x, y, x+w, y+h])
            labels.append(cat_id_to_label[obj['category_id']])

        targets_formatted.append({
            "boxes": torch.tensor(boxes),
            "labels": torch.tensor(labels)
        })

    metric.update(preds_formatted, targets_formatted)

result = metric.compute()
print(result)

{'map': tensor(0.0478), 'map_50': tensor(0.0478), 'map_75': tensor(-1.), 'map_small': tensor(-1.), 'map_medium': tensor(0.0842), 'map_large': tensor(0.0500), 'mar_1': tensor(0.), 'mar_10': tensor(0.3000), 'mar_100': tensor(0.3000), 'mar_small': tensor(-1.), 'mar_medium': tensor(0.2500), 'mar_large': tensor(0.2500), 'map_per_class': tensor(-1.), 'mar_100_per_class': tensor(-1.), 'classes': tensor([ 1,  2,  3,  4,  5,  6, 64, 67], dtype=torch.int32)}


Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.


In [ ]:
result['map_50']

tensor(0.0478)

🎯 5. F1-Score

In [ ]:
precision = result['map_50']
recall = result['mar_100']

f1 = 2 * (precision * recall) / (precision + recall)
print(f"F1-score: {f1}")

F1-score: 0.08240238577127457


# **Código Completo**

In [ ]:
!pip install roboflow torch torchvision matplotlib opencv-python

from roboflow import Roboflow
rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(1)
dataset = version.download("coco")

import torch
import torchvision
from torchvision.datasets import CocoDetection
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = CocoDetection(
    root=dataset.location + "/train",
    annFile=dataset.location + "/train/_annotations.coco.json",
    transform=transform
)

val_dataset = CocoDetection(
    root=dataset.location + "/valid",
    annFile=dataset.location + "/valid/_annotations.coco.json",
    transform=transform
)

from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

import torchvision.models.detection as detection

num_classes = len(train_dataset.coco.getCatIds()) + 1  # +1 background

model = detection.ssdlite320_mobilenet_v3_large(pretrained=True)
model.head.classification_head.num_classes = num_classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

import torch.optim as optim

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params, lr=0.0003)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]

        targets_formatted = []
        for t in targets:
            boxes = []
            labels = []

            for obj in t:
                x, y, w, h = obj['bbox']
                boxes.append([x, y, x+w, y+h])
                labels.append(obj['category_id'])

            targets_formatted.append({
                "boxes": torch.tensor(boxes, dtype=torch.float32).to(device),
                "labels": torch.tensor(labels, dtype=torch.int64).to(device)
            })

        loss_dict = model(images, targets_formatted)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

model.eval()

images, _ = next(iter(val_loader))
images = [img.to(device) for img in images]

with torch.no_grad():
    predictions = model(images)

print(predictions[0])

torch.save(model.state_dict(), "ssd_mobilenetv3.pth")

model.eval()

scripted_model = torch.jit.script(model)
scripted_model.save("ssd_mobilenetv3_mobile.pt")

loading Roboflow workspace...
loading Roboflow project...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1`. You can also use `weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.


Epoch 1, Loss: 13.354667000148607
Epoch 2, Loss: 9.690548026043436
Epoch 3, Loss: 8.742304739744768
Epoch 4, Loss: 7.642176856165347
Epoch 5, Loss: 6.347276832746423
Epoch 6, Loss: 5.373516652895057
Epoch 7, Loss: 4.418159287908803
Epoch 8, Loss: 3.5608914935070537
Epoch 9, Loss: 2.8323848143867822
Epoch 10, Loss: 2.544353267420893
Epoch 11, Loss: 2.2248550238816636
Epoch 12, Loss: 2.0497947153837783
Epoch 13, Loss: 1.8841495565746142
Epoch 14, Loss: 1.6411476238914158
Epoch 15, Loss: 1.522485225096993
Epoch 16, Loss: 1.424339781636777
Epoch 17, Loss: 1.2743800541628962
Epoch 18, Loss: 1.2295586518619372
Epoch 19, Loss: 1.1205617137577222
Epoch 20, Loss: 1.015472922636115


KeyboardInterrupt: 

# **Testes do Modelo**

converter seu modelo SSD + MobileNetV3 para TensorFlow Lite (TFLite)

🚀 PIPELINE COMPLETO

```
PyTorch (.pth)
   ↓
ONNX (.onnx)
   ↓
TensorFlow (.pb)
   ↓
TensorFlow Lite (.tflite)
```



⚙️ 1. Instalar dependências no Colab

In [ ]:
!pip install onnx onnxruntime onnx-tf tensorflow

📤 2. Exportar PyTorch → ONNX

⚠️ Aqui está o ponto crítico: modelos de detecção precisam ajuste

In [ ]:
import torch

model.eval()

dummy_input = [torch.rand(3, 320, 320).to(device)]

torch.onnx.export(
    model,
    (dummy_input,),
    "model.onnx",
    opset_version=11,
    input_names=["images"],
    output_names=["output"],
    dynamic_axes=None
)

🔥 SOLUÇÃO MAIS ESTÁVEL (RECOMENDADO)

Converter usando um wrapper:

In [ ]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        outputs = self.model(x)
        # pegar apenas boxes (simplifica export)
        return outputs[0]['boxes']

wrapped_model = ModelWrapper(model)
wrapped_model.eval()

dummy_input = [torch.rand(3, 320, 320).to(device)]

torch.onnx.export(
    wrapped_model,
    (dummy_input,),
    "model.onnx",
    opset_version=11
)

🔄 3. ONNX → TensorFlow

In [ ]:
from onnx_tf.backend import prepare
import onnx

onnx_model = onnx.load("model.onnx")
tf_rep = prepare(onnx_model)

tf_rep.export_graph("model_tf")

📱 4. TensorFlow → TFLite

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model("model_tf")

# otimização (IMPORTANTE pra mobile)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

✅ 5. Testar modelo TFLite

In [ ]:
interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(input_details)
print(output_details)